# Pose clarity check: does denoising or a larger input expose more of a person's body?

**Question (owner, 2026-11-05).** Posture detection looked doubtful because UCF-Crime footage is small and unclear. Would
it work better if the frames were denoised first, or fed to the pose model at a larger size?

**What this measures.** For the same frames, the same people, and the same YOLO11m-pose settings as the cache
(`conf=0.25, iou=0.7, person only`), the share of people whose *shoulder, hip and knee* the detector sees at
confidence >= 0.5 ("usable body", the definition in the 2026-11-05 posture review). Five conditions, one change each:

| Condition | What changes versus `orig_640` |
|---|---|
| `orig_640` | nothing (re-run of the cached setting; must agree with the cache) |
| `orig_960`, `orig_1280` | the detector's input size only; no pixels are added by us |
| `restormer_640` | frame denoised by Restormer (blind Gaussian, D-62), then pose at 640 |
| `scunet_640` | frame denoised by SCUNet real-PSNR (D-62), then pose at 640 |

NAFNet is left out (D-61: worst restorer, diverges at high noise) and so is adaptive median (it fixes salt-and-pepper
noise, which UCF frames do not have).

**What this does not measure.** Pose *accuracy*. Usable-body rate is how much body the detector *claims* to see; a
restorer can raise confidence without adding truth (rule 2: model output is never ground truth). Part 6 writes
side-by-side skeleton sheets for a human look, and the verdict is only a measurement until that look is done.

**Known bias, stated up front.** Reference people are those the original frame detected, which favours `orig_640` on
detection. People that other conditions find on frames where the cache found nobody are reported separately (Part 5).

**Decision rule, fixed before running** (the 0.05 threshold is our proposal, not from a source): a condition *helps* only
if usable-body rate rises by >= 0.05 over `orig_640`, the paired 95% interval (videos resampled, not frames) excludes
zero, and the human look shows no rise in failed skeletons. Otherwise the result is recorded as inside the noise or negative.

In [ ]:
import json, subprocess, sys, time
from pathlib import Path
import numpy as np, pandas as pd, cv2, torch

REPO = Path.cwd()
while not (REPO / "core").exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path[:0] = [str(REPO), str(REPO / "scripts")]
from core import pose_clarity as pc
from core.provenance import sha256_path
from extract_p1c_video_frames import encode_jpeg_rgb, decode_jpeg_bytes   # JPEG q95 step every cached frame went through (K-33)

CONFIG = dict(
    seed=20261105, per_bucket=60, n_empty=60, n_boot=2000,
    conf=0.25, iou=0.7, jpeg_quality=95, sheet_frames=12,
    conditions={"orig_640": dict(restorer=None, imgsz=640), "orig_960": dict(restorer=None, imgsz=960),
                "orig_1280": dict(restorer=None, imgsz=1280), "restormer_640": dict(restorer="restormer_gauss_blind", imgsz=640),
                "scunet_640": dict(restorer="scunet_real_psnr", imgsz=640)},
)
OUT = REPO / "results/p1c/pose-clarity-20261105"; OUT.mkdir(parents=True, exist_ok=True)
DETECTIONS = REPO / "results/p1c/p1c-roi-cache/detections.jsonl"
VIDEO_ROOT = REPO / "Datasets/UCF Crime Official/raw/UCF_Crimes/Videos"
YOLO_W = REPO / ".cache/weights/yolo11m-pose-29b17eaf.pt"
RESTORER_CFG = REPO / "configs/p2b_extra_restorers.yaml"
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("device:", device, "| repo:", REPO)

## 1. Plan: which frames and which people

In [ ]:
records = [json.loads(l) for l in open(DETECTIONS)]
plan = pc.plan_frames(records, CONFIG["per_bucket"], CONFIG["n_empty"], CONFIG["seed"])
plan_df = pd.DataFrame([{k: p[k] for k in ("video", "frame", "stratum")} | {"ref_people": len(p["ref_persons"])} for p in plan])
print(len(plan), "frames,", int(plan_df.ref_people.sum()), "reference people")
plan_df.groupby("stratum").agg(frames=("frame", "size"), people=("ref_people", "sum"))

## 2. Decode the exact frames
Videos are read **sequentially** (seeking is unreliable on H.264, see `scripts/extract_p1c_video_frames.py`), and every kept
frame goes through the same JPEG quality-95 round trip as the cached frames.

In [ ]:
paths = {}
for p in VIDEO_ROOT.rglob("*.mp4"):
    paths.setdefault(p.stem, p)
need = {}
for p in plan:
    need.setdefault(p["video"], set()).add(p["frame"])
frames = {}   # (video, frame) -> RGB uint8
t0 = time.time()
for video, wanted in need.items():
    cap, i, last = cv2.VideoCapture(str(paths[video])), 0, max(wanted)
    while i <= last:
        ok, bgr = cap.read()
        if not ok: break
        if i in wanted:
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            frames[(video, i)] = decode_jpeg_bytes(encode_jpeg_rgb(rgb, quality=CONFIG["jpeg_quality"]))
        i += 1
    cap.release()
missing = [(p["video"], p["frame"]) for p in plan if (p["video"], p["frame"]) not in frames]
print(len(frames), "frames decoded in", round(time.time() - t0), "s; missing:", missing)
assert not missing, "some planned frames could not be decoded"

## 3. Build the five inputs and run pose on each

In [ ]:
from ultralytics import YOLO
from models.restoration_comparator import load_restorers
assert sha256_path(YOLO_W) == "29b17eaf3a3117cbea906090dbedf9159f7c6a49db58ec8b99ed2dfde1cf6eb2", "pose weights differ from the registered ones"
pose = YOLO(str(YOLO_W))
restorers = {n: (r, prov) for n, r, prov in load_restorers(RESTORER_CFG, REPO, device)}

keys = [(p["video"], p["frame"]) for p in plan]
def restore_all(restorer):
    out = {}
    with torch.no_grad():
        for s in range(0, len(keys), 16):
            chunk = keys[s:s + 16]
            x = torch.from_numpy(np.stack([frames[k] for k in chunk])).permute(0, 3, 1, 2).float().div(255).to(device)
            y = restorer.restore(x).mul(255).round().clamp(0, 255).byte().permute(0, 2, 3, 1).cpu().numpy()
            out.update({k: y[i] for i, k in enumerate(chunk)})
    return out

inputs = {}
for name, spec in CONFIG["conditions"].items():
    inputs[name] = frames if spec["restorer"] is None else restore_all(restorers[spec["restorer"]][0])

def detect(images, imgsz):
    found = {}
    for s in range(0, len(keys), 32):
        chunk = keys[s:s + 32]
        res = pose.predict([np.ascontiguousarray(images[k][..., ::-1]) for k in chunk], imgsz=imgsz, conf=CONFIG["conf"],
                           iou=CONFIG["iou"], classes=[0], verbose=False)
        for k, r in zip(chunk, res):
            xyxy, kps = r.boxes.xyxy.cpu().numpy(), r.keypoints.data.cpu().numpy()
            found[k] = [{"box": xyxy[j].tolist(), "keypoints": kps[j].tolist()} for j in range(len(xyxy))]
    return found

detections = {}
for name, spec in CONFIG["conditions"].items():
    t0 = time.time(); detections[name] = detect(inputs[name], spec["imgsz"]); print(name, round(time.time() - t0, 1), "s")

## 4. Usable-body rate on the reference people (the main result)
Each reference person is matched to the nearest box of the condition's own run (IoU >= 0.5). A person the condition does
not find counts as not usable. Intervals resample **videos**.

In [ ]:
ref_plan = [p for p in plan if p["ref_persons"]]
per_video = {c: {} for c in CONFIG["conditions"]}   # condition -> video -> rows
for c in CONFIG["conditions"]:
    for p in ref_plan:
        per_video[c][p["video"]] = pc.score_condition(p["ref_persons"], detections[c][(p["video"], p["frame"])])

def flat(c, bucket=None):
    return [r for rows in per_video[c].values() for r in rows if bucket in (None, r["bucket"])]

table = []
for c in CONFIG["conditions"]:
    rows = flat(c)
    b = pc.paired_video_bootstrap(per_video[c], per_video["orig_640"], n_boot=CONFIG["n_boot"], seed=CONFIG["seed"])
    j = pc.paired_video_bootstrap(per_video[c], per_video["orig_640"], key="joints", n_boot=CONFIG["n_boot"], seed=CONFIG["seed"])
    table.append(dict(condition=c, people=len(rows), found=np.mean([r["matched"] for r in rows]), usable=np.mean([r["usable"] for r in rows]),
                      mean_joints=np.mean([r["joints"] for r in rows]),
                      usable_diff=b["diff"], ci_lo=b["lo"], ci_hi=b["hi"], joints_diff=j["diff"],
                      verdict="(baseline)" if c == "orig_640" else pc.verdict(b)))
summary = pd.DataFrame(table).round(3)
summary

In [ ]:
by_bucket = pd.DataFrame([{ "condition": c, **{name: round(np.mean([r["usable"] for r in flat(c, name)]), 3) for _, _, name in pc.BUCKETS if flat(c, name)}}
                          for c in CONFIG["conditions"]]).set_index("condition")
by_bucket   # usable-body rate by person height (px); small buckets are where a gain would have to show up

## 5. Extra people, and agreement with the cache
Frames where the cache found nobody: how many people (with a usable body) does each condition report? Some will be false
positives, which is why Part 6 exists.

In [ ]:
empty = [p for p in plan if not p["ref_persons"]]
extra = pd.DataFrame([{"condition": c,
                       "people_found": sum(len(detections[c][(p["video"], p["frame"])]) for p in empty),
                       "with_usable_body": sum(pc.usable_body(d["keypoints"]) for p in empty for d in detections[c][(p["video"], p["frame"])])}
                      for c in CONFIG["conditions"]]).set_index("condition")
print(len(empty), "frames with no cached person")
display(extra)
# orig_640 against the cache: if this is not ~1.0 the frames or settings differ from the cached run, and nothing above is comparable.
agree = np.mean([r["matched"] for r in flat("orig_640")])
print("orig_640 re-finds", round(float(agree), 3), "of the cached people (expect close to 1.0)")

## 6. Look at it (human check, not ground truth)
Skeletons for `orig_640` and each other condition on the same frames, drawn at 2x. Judge only whether the skeleton follows
the visible body. Count failures per condition by hand and write the counts into the report; a higher usable-body rate with
more wrong skeletons is not a gain.

In [ ]:
EDGES = [(5,6),(5,7),(7,9),(6,8),(8,10),(5,11),(6,12),(11,12),(11,13),(13,15),(12,14),(14,16)]
def overlay(img, persons, s=2):
    im = cv2.resize(img, None, fx=s, fy=s, interpolation=cv2.INTER_NEAREST).copy()
    for p in persons:
        k = p["keypoints"]
        for a, b in EDGES:
            if k[a][2] >= .3 and k[b][2] >= .3:
                cv2.line(im, (int(k[a][0]*s), int(k[a][1]*s)), (int(k[b][0]*s), int(k[b][1]*s)), (0,255,0), 1, cv2.LINE_AA)
        for x, y, c in k[5:]:
            if c >= .3: cv2.circle(im, (int(x*s), int(y*s)), 2, (255,0,0), -1)
    return im
rng = np.random.default_rng(CONFIG["seed"])
pick = [ref_plan[i] for i in rng.choice(len(ref_plan), CONFIG["sheet_frames"], replace=False)]
names = list(CONFIG["conditions"])
rows_img = []
for p in pick:
    k = (p["video"], p["frame"])
    tiles = [cv2.putText(overlay(inputs[c][k], detections[c][k]), f"{c}  {p['video'][:14]} f{p['frame']}", (4, 14),
                         cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255,255,0), 1, cv2.LINE_AA) for c in names]
    rows_img.append(np.concatenate(tiles, axis=1))
for i in range(0, len(rows_img), 3):
    sheet = np.concatenate(rows_img[i:i+3], axis=0)
    path = OUT / f"sheet_{i // 3 + 1}.jpg"; cv2.imwrite(str(path), sheet[..., ::-1])
    display(__import__("PIL.Image", fromlist=["Image"]).fromarray(sheet))
print("sheets written to", OUT, "(JPEGs are licensed video derivatives: git-ignored, keep local)")

## 7. Save the result with its provenance

In [ ]:
def git(*a): return subprocess.check_output(["git", *a], cwd=REPO, text=True).strip()
record = dict(
    question="Does denoising or a larger input size expose more usable body in UCF-Crime frames?",
    config=CONFIG, seed=CONFIG["seed"],
    code_revision=git("rev-parse", "HEAD"), working_tree_dirty=bool(git("status", "--porcelain", "--", "core", "models", "scripts", "notebooks")),
    data=dict(detections_sha256=sha256_path(DETECTIONS), pose_weights_sha256=sha256_path(YOLO_W),
              restorer_config_sha256=next(iter(restorers.values()))[1]["config_sha256"],
              restorer_weights_sha256={n: r[1].get("weights_sha256") for n, r in restorers.items()}),
    frames=len(plan), reference_people=int(plan_df.ref_people.sum()),
    summary=summary.to_dict("records"), usable_by_height=by_bucket.reset_index().to_dict("records"),
    extra_people_on_empty_frames=extra.reset_index().to_dict("records"), orig_640_cache_agreement=float(agree),
    human_check=dict(done=False, note="fill in failure counts per condition after looking at the sheets"),
    labels="usable-body rate is detector confidence, not pose accuracy and not ground truth",
)
(OUT / "summary.json").write_text(json.dumps(record, indent=2))
print("wrote", OUT / "summary.json")